# EDA A1 – Jakość danych (wersja pandas)

Pytanie biznesowe: **czy możemy ufać tym danym?**

Pracujemy na tabelach `users`, `orders`, `order_items`, `products`, `inventory_items` i `events`
z `bigquery-public-data.thelook_ecommerce`.

* Wczytaj tylko potrzebne kolumny i zamień je na najmniejsze odpowiednie typy danych.
  Sprawdź zużycie pamięci.
* Czy w danych są braki? Które są oczekiwane (proces biznesowy), a które są problemem?
* Czy klucze główne są unikalne (duplikaty)?
* Czy znaczniki czasu zgadzają się ze statusami zamówień?
* Jak świeże są dane i czy są rekordy z przyszłymi datami?
* Czy wartości liczbowe (ceny, koszty) mają sens?

In [ ]:
import pandas as pd
import numpy as np

from bq_utils import read_bq

## Wczytanie danych

* `read_bq()` działa jak `pd.read_csv(usecols=..., dtype=...)`, tylko czyta z BigQuery.
* Pomijamy kolumny z danymi osobowymi (imię, nazwisko, e-mail, adres) – nie są nam potrzebne,
  a dobry nawyk to nie pobierać danych osobowych "na zapas".
* `Int32` / `Int8` to typy całkowite, które dopuszczają braki (NULL); `category` – dla tekstu,
  który powtarza się wiele razy (status, kraj, kategoria).

In [ ]:
# Users – specify columns to keep and datatypes to convert

users_cols = ["id", "age", "gender", "country", "city", "traffic_source", "created_at"]

users_dtypes = {"id": "Int32", "age": "Int8", "gender": "category",
                "country": "category", "city": "category", "traffic_source": "category"}

users = read_bq("users", usecols=users_cols, dtype=users_dtypes)

users.head()

In [ ]:
users.info(memory_usage="deep")

In [ ]:
# Orders

orders_cols = ["order_id", "user_id", "status", "gender", "num_of_item",
               "created_at", "shipped_at", "delivered_at", "returned_at"]

orders_dtypes = {"order_id": "Int32", "user_id": "Int32", "status": "category",
                 "gender": "category", "num_of_item": "Int8"}

orders = read_bq("orders", usecols=orders_cols, dtype=orders_dtypes)

orders.info(memory_usage="deep")

In [ ]:
# Order items

order_items_cols = ["id", "order_id", "user_id", "product_id", "inventory_item_id", "status",
                    "sale_price", "created_at", "shipped_at", "delivered_at", "returned_at"]

order_items_dtypes = {"id": "Int32", "order_id": "Int32", "user_id": "Int32",
                      "product_id": "Int32", "inventory_item_id": "Int32", "status": "category"}

order_items = read_bq("order_items", usecols=order_items_cols, dtype=order_items_dtypes)

order_items.info(memory_usage="deep")

In [ ]:
# Products – "name" zostawiamy jako tekst: prawie każda nazwa jest inna,
# więc typ category nic by nie oszczędził (category opłaca się przy wielu powtórzeniach)

products_cols = ["id", "name", "brand", "category", "department",
                 "cost", "retail_price", "distribution_center_id"]

products_dtypes = {"id": "Int32", "brand": "category", "category": "category",
                   "department": "category", "distribution_center_id": "Int8"}

products = read_bq("products", usecols=products_cols, dtype=products_dtypes)

products.info(memory_usage="deep")

In [ ]:
# Inventory items

inventory_cols = ["id", "product_id", "created_at", "sold_at", "cost",
                  "product_category", "product_department"]

inventory_dtypes = {"id": "Int32", "product_id": "Int32",
                    "product_category": "category", "product_department": "category"}

inventory = read_bq("inventory_items", usecols=inventory_cols, dtype=inventory_dtypes)

inventory.info(memory_usage="deep")

In [ ]:
# Events – 2,4 mln wierszy, więc wczytujemy 10% próbkę (TABLESAMPLE).
# Wnioski z tej tabeli to SZACUNKI – pełne liczby policzyliśmy wcześniej w SQL.

events_cols = ["id", "user_id", "session_id", "sequence_number", "created_at",
               "traffic_source", "event_type", "browser"]

events_dtypes = {"id": "Int32", "user_id": "Int32", "sequence_number": "Int16",
                 "traffic_source": "category", "event_type": "category", "browser": "category"}

events = read_bq("events", usecols=events_cols, dtype=events_dtypes, sample_pct=10)

events.info(memory_usage="deep")

In [ ]:
# Store all tables in a dictionary – kolejne sprawdzenia robimy dla wszystkich tabel naraz

tables = {"users": users, "orders": orders, "order_items": order_items,
          "products": products, "inventory": inventory, "events (10%)": events}

## Braki danych

* Policz % braków w każdej kolumnie każdej tabeli.
* Pokaż tylko kolumny, w których braki występują, od największego udziału.

In [ ]:
# Check missing value percentage
# pd.concat ze słownikiem tworzy jedną Series z dwupoziomowym indeksem (tabela, kolumna)

(pd.concat({name: df.isna().mean().mul(100) for name, df in tables.items()})
 .rename("null_pct")
 .to_frame()
 .query("null_pct > 0")
 .sort_values("null_pct", ascending=False)
 .round(2)
)

## Duplikaty klucza głównego

* Czy każdy klucz występuje dokładnie raz? Uwaga: w `orders` kluczem jest `order_id`, nie `id`.

In [ ]:
# Compare row count with number of unique keys

keys = {"users": "id", "orders": "order_id", "order_items": "id",
        "products": "id", "inventory": "id", "events (10%)": "id"}

(pd.DataFrame({"rows": {name: len(df) for name, df in tables.items()},
               "unique_keys": {name: tables[name][key].nunique() for name, key in keys.items()}})
 .assign(duplicates=lambda x: x["rows"] - x["unique_keys"])
)

## Spójność statusów i znaczników czasu

* Dla każdego statusu zamówienia policz, jaki odsetek zamówień ma `shipped_at`, `delivered_at`
  i `returned_at`. Spodziewamy się np., że "Processing" nie ma żadnej daty wysyłki,
  a "Returned" ma wszystkie trzy.

In [ ]:
# Create boolean flags with assign, then average them by status (mean of True/False = % True)

(orders
 .assign(shipped=lambda x: x["shipped_at"].notna(),
         delivered=lambda x: x["delivered_at"].notna(),
         returned=lambda x: x["returned_at"].notna())
 .groupby("status", observed=True)
 .agg({"order_id": "count", "shipped": "mean", "delivered": "mean", "returned": "mean"})
 .rename(columns={"order_id": "orders"})
 .sort_values("orders", ascending=False)
 .round(2)
)

## Świeżość danych i daty z przyszłości

* Dla każdej tabeli z `created_at` znajdź najstarszy i najnowszy rekord.
* Policz rekordy z datą późniejszą niż "teraz" i o ile dni wybiegają w przyszłość.

In [ ]:
now = pd.Timestamp.now(tz="UTC")

dated = {name: df for name, df in tables.items() if "created_at" in df.columns}

created = pd.concat({name: df["created_at"] for name, df in dated.items()}, names=["table", None])

freshness = (created
             .groupby(level="table")
             .agg(oldest="min",
                  newest="max",
                  future_rows=lambda s: (s > now).sum())
             .assign(days_ahead=lambda x: (x["newest"] - now).dt.days)
            )

freshness

In [ ]:
# Inspect a few future-dated order items – use @ to reference a Python variable inside query

order_items.query("created_at > @now").sort_values("created_at").head()

## Wartości liczbowe

* Sprawdź rozkład cen i kosztów z `describe()`.
* Czy są produkty, których koszt jest wyższy niż cena katalogowa (sprzedaż ze stratą)?
* Czy są pozycje sprzedane za 0 lub mniej?

In [ ]:
products[["cost", "retail_price"]].describe().round(2)

In [ ]:
order_items["sale_price"].describe().round(2)

In [ ]:
# Products sold at a loss and items with non-positive price

products.query("cost > retail_price").shape[0], order_items.query("sale_price <= 0").shape[0]

In [ ]:
# Margin by department – first business-flavoured view of the data

(products
 .assign(margin_pct=lambda x: (x["retail_price"] - x["cost"]) / x["retail_price"] * 100)
 .groupby("department", observed=True)
 .agg({"id": "count", "retail_price": "mean", "margin_pct": "mean"})
 .rename(columns={"id": "products"})
 .round(1)
)

## Wnioski (uzupełnij po uruchomieniu)

* Braki: które są oczekiwane (wynikają z procesu), a które są problemem jakości?
* Duplikaty: czy klucze są unikalne?
* Spójność: czy daty zgadzają się ze statusami?
* Świeżość: ile tabel ma rekordy z przyszłości i o ile dni?
* Wartości: czy ceny i marże mają sens?

Wnioski z wersji SQL są zapisane w CLAUDE.md (sekcja "Wnioski A1").